# Phase 3: Fine-tune DistilBERT (Dark Pattern Text Detector)
Run the cells **top to bottom**. Before starting: **Runtime → Change runtime type → T4 GPU**.
Training takes roughly 5-15 minutes. At the end, download the model zip and copy the printed results to Claude.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("No GPU! Use Runtime > Change runtime type > T4 GPU, then run again.")

## 1. Get the code and data from GitHub

In [ ]:
%cd /content
!rm -rf dark-pattern-text-detector
!git clone https://github.com/AmDgoat/dark-pattern-text-detector.git
%cd dark-pattern-text-detector
!pip -q install transformers accelerate scikit-learn pandas
!ls data/processed

## 2. Train and evaluate (uses the committed train/val/test splits)

In [ ]:
!python src/train_distilbert.py --out_dir /content/dp_model

## 3. Results summary (copy this output and send it to Claude)

In [ ]:
import json
m = json.load(open("results/distilbert_metrics.json"))
print("Test accuracy        :", round(m["test_accuracy"], 4))
print("Test macro F1        :", round(m["test_macro_f1"], 4))
print("Dark vs not-dark acc :", round(m["binary_accuracy"], 4))
print("Dark class F1        :", round(m["binary_f1_dark"], 4))
print()
for lab in m["labels"]:
    r = m["category_report"][lab]
    print(f'{lab:14s} P={r["precision"]:.3f} R={r["recall"]:.3f} F1={r["f1-score"]:.3f} n={int(r["support"])}')
print()
print("Confusion matrix (rows=true, cols=pred):", m["labels"])
for row in m["confusion_matrix"]:
    print(row)

## 4. Look at the mistakes (useful for the report)

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.read_csv("results/distilbert_test_errors.csv").head(25)

## 5. Download the model
The model is about 250 MB, so it **must not** go on GitHub. Download it and keep it on your laptop
(for example in a folder called `models/`). The small results files can go to GitHub.

In [ ]:
!cd /content/dp_model && zip -rq /content/dp_model_final.zip final
from google.colab import files
files.download("/content/dp_model_final.zip")

In [ ]:
# Also download the small results files (upload these to GitHub results/ folder)
!zip -q /content/results_distilbert.zip results/distilbert_metrics.json results/distilbert_test_errors.csv
files.download("/content/results_distilbert.zip")